# LLM Request SDK Quickstart

서버화된 `llm-request`를 SDK로 사용하는 가이드.

**구조:**
```
이 노트북 (SDK)                     llm-request Service (port 8101)
┌──────────────────┐                ┌─────────────────────────┐
│ checksum 계산     │   HTTP        │ MDB 캐시 조회 (LLM 응답) │
│ Redis 파일캐시    │ ───────────→  │ 파일 처리 (Files API)    │
│ 서버 호출         │               │ LLM 호출 (Gemini)       │
│ 결과 수신         │ ←─────────── │ 결과 저장 + 반환         │
└──────────────────┘                └─────────────────────────┘
```

**기존 라이브러리와 차이점:**
- 설정: handler URL 3개 → `LLM_REQUEST_URL` 1개
- 사용 코드: 동일 (`llm_request_async(...)`)
- 내부 동작: 직접 실행 → HTTP로 서버에 위임

## 설정

In [ ]:
import os

# SDK는 서비스 URL 하나만 설정하면 됨
os.environ["LLM_REQUEST_URL"] = "http://192.7.9.45:8101"

# Redis (SDK에서 파일 캐시 조회용)
os.environ["REDIS_URL"] = "redis://192.7.9.45:7617/0"

from llm_request import configure
configure()

## 타이밍 유틸리티

각 단계별 소요 시간과 네트워크 흐름을 추적하는 헬퍼.

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer(label: str):
    """구간별 소요 시간 측정."""
    start = time.perf_counter()
    print(f"  [{label}] 시작...")
    yield
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  [{label}] 완료: {elapsed:.0f}ms")

def print_flow(title: str, meta: dict, elapsed_ms: float):
    """요청 흐름 요약 출력."""
    source = meta.get("source", "?")
    cache_hit = meta.get("cache_hit", False)
    cache_key = meta.get("cache_key", "")[:16]
    
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  소스:      {source} ({'MDB 캐시 히트' if cache_hit else 'LLM 호출'})")
    print(f"  캐시 키:   {cache_key}...")
    print(f"  총 소요:   {elapsed_ms:.0f}ms")
    
    if meta.get("route_reason"):
        print(f"  경로:      {meta['route_reason']}")
    print()

---
## 1. 기본 호출 — 텍스트만 (파일 없음)

```
SDK                              Service (8101)
 │  POST /v1/request             │
 │  {llm_input, cache_mode}  ──→ │ → MDB 캐시 조회
 │                               │ → MISS → LLM 호출 (Gemini)
 │  ←── {result, meta}          │ → MDB 캐시 저장
```

In [ ]:
from llm_request import llm_request_async

# --- 1회차: LLM 직접 호출 (캐시 miss) ---
start = time.perf_counter()
result, meta = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("1회차: 첫 호출 (캐시 miss → LLM)", meta, elapsed)
print(f"  응답: {result.get('text', '')[:100]}")

In [ ]:
# --- 2회차: 동일 입력 → MDB 캐시 히트 (LLM 호출 없음) ---
start = time.perf_counter()
result2, meta2 = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed2 = (time.perf_counter() - start) * 1000

print_flow("2회차: 동일 입력 (MDB 캐시 히트)", meta2, elapsed2)
print(f"  응답: {result2.get('text', '')[:100]}")
print(f"  속도 비교: {elapsed:.0f}ms → {elapsed2:.0f}ms ({elapsed/max(elapsed2,0.1):.1f}x 빠름)")

---
## 2. 파일 포함 호출

```
SDK (로컬)                              Service (8101)
 │                                       │
 │  1. checksum 계산 (로컬 SHA-256)       │
 │  2. Redis 파일 캐시 조회 (로컬)         │
 │     miss → POST /v1/files             │
 │     (파일 바이너리 전송)           ──→  │ → Files API 업로드
 │                                       │ → Redis 파일 캐시 저장
 │  3. POST /v1/request                  │
 │     (checksum만 전송)             ──→  │ → MDB 캐시 조회
 │                                       │ → MISS → LLM 호출
 │  ←── {result, meta}                  │ → MDB 캐시 저장
```

In [ ]:
from pathlib import Path
from llm_request import llm_request_async
from llm_request.checksum import file_checksum

# 테스트용 파일 (quickstart.pdf 또는 아무 PDF)
test_file = "quickstart.pdf"
if not Path(test_file).exists():
    print(f"⚠️ {test_file}이 없습니다. 아래 셀은 건너뛰세요.")
else:
    # --- 단계별 타이밍 측정 ---
    print(f"파일: {test_file} ({Path(test_file).stat().st_size / 1024:.0f}KB)")
    print()

    # 1. checksum 계산 (로컬)
    with timer("checksum 계산 (로컬)"):
        cs = file_checksum(test_file)
    print(f"    → {cs['checksum'][:24]}... ({cs['mime_type']})")
    print()

    # 2. 전체 호출 (SDK가 내부에서 Redis 조회 + 파일 업로드 + 서버 요청)
    with timer("llm_request_async 전체"):
        result, meta = await llm_request_async(
            {
                "contents": "이 문서를 한 문장으로 요약해줘",
                "model": "gemini-2.5-flash",
            },
            file_paths=[test_file],
            database="kw_ai_dev",
        )
    
    print()
    print_flow("파일 포함 호출 (1회차)", meta, 0)
    print(f"  응답: {result.get('text', '')[:200]}")

---
## 3. done_check + fetch_by_cache_key

캐시 존재 여부만 확인하고, 필요 시 직접 조회.

```
SDK                              Service (8101)
 │  POST /v1/done-check          │
 │  {llm_input}             ──→  │ → MDB 캐시 조회
 │  ←── {cache_key, cached}     │
 │                               │
 │  POST /v1/fetch               │
 │  {cache_key}             ──→  │ → MDB에서 결과 반환
 │  ←── {result, meta}          │
```

In [ ]:
from llm_request import done_check, fetch_by_cache_key

# --- done_check: 캐시 존재 여부 확인 ---
with timer("done_check"):
    dc = await done_check(
        {"contents": "1+1은?", "model": "gemini-2.5-flash"},
        database="kw_ai_dev",
    )

print(f"  cache_key: {dc['cache_key'][:16]}...")
print(f"  cache_hit: {dc['cache_hit']}")
print()

# --- fetch_by_cache_key: 캐시 키로 직접 조회 ---
if dc["cache_hit"]:
    with timer("fetch_by_cache_key"):
        result, meta = await fetch_by_cache_key(
            dc["cache_key"],
            database="kw_ai_dev",
        )
    print(f"  응답: {result.get('text', '')[:100]}")
else:
    print("  캐시 없음 — 위의 1번 셀을 먼저 실행하세요")

---
## 4. 기존 라이브러리와 비교

| | 라이브러리 (v1) | SDK (v2, 서버화) |
|--|---------------|----------------|
| **설정** | MDB_HANDLER_URL + LLM_HANDLER_URL + REDIS_URL + configure 3개 | LLM_REQUEST_URL 1개 |
| **의존성** | llm-handler, mdb-handler, redis-handler 전부 | httpx, redis-handler만 |
| **사용 코드** | `llm_request_async(...)` | `llm_request_async(...)` (동일) |
| **파일 처리** | 로컬에서 직접 Files API 호출 | checksum 계산(로컬) → 서버가 처리 |
| **캐시 조회** | 로컬에서 직접 MDB 조회 | 서버가 조회 |